<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module3/m3_l3_nb2_anomaly_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Module 3 · Lesson 3 · Notebook 2
# Anomaly Detection with Autoencoders

---

**Module 3 · Introduction to Autoencoders**  
**Estimated time:** 35 minutes  
**This notebook is optional.** Complete Notebook 1 (Denoising) first.  
**Prerequisites:** Lesson 2 and Lesson 3 Notebook 1 complete

---

## 📋 What This Notebook Does

Autoencoders trained on normal data cannot reconstruct anomalies accurately. This is not a failure — it is the detection mechanism. This notebook demonstrates that mechanism through two progressively realistic activities:

**Part A — Synthetic 2D Data (~15 min):** A minimal, fast demonstration of the core idea using 2D points. Normal data forms a cluster; anomalies are scattered outside it. The reconstruction error scatter plot makes the mechanism visually unmistakable.

**Part B — MNIST One-Class Detection (~20 min):** A more realistic scenario. You train the autoencoder only on images of one digit class (e.g., all the 0s), then test it on all digit classes. The digits it was never trained on become the anomalies.

By the end of this notebook you will have:

- Implemented reconstruction-based anomaly detection from scratch on synthetic data
- Trained a one-class autoencoder on MNIST and evaluated its anomaly scores across all digit classes
- Set and evaluated a detection threshold — measuring precision and recall of the anomaly detector
- Engaged with the honest limitations of the approach through guided discussion questions

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Imports and preprocessing |
| **Part A.1** | Synthetic dataset — normal cluster + outliers |
| **Part A.2** | Train a tiny autoencoder on normal data only |
| **Part A.3** | Reconstruction error as anomaly score |
| **Part A.4** | Set a threshold and evaluate |
| **Part B.1** | MNIST one-class setup — select normal class |
| **Part B.2** | Train on normal class only |
| **Part B.3** | Reconstruction error across all digit classes |
| **Part B.4** | ROC curve and AUC |
| **Part B.5** | Limitations discussion |

---

> **The core logic throughout:** Train on normal. Test on everything. High reconstruction error = the model cannot represent this input = possibly anomalous.

---
## Step 0 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import roc_curve, auc, precision_recall_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

np.random.seed(42)
tf.random.set_seed(42)

# Load and preprocess MNIST for Part B
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.mnist.load_data()
x_train_all = x_train_raw.astype('float32') / 255.0
x_test_all  = x_test_raw.astype('float32')  / 255.0
x_train_all = x_train_all.reshape(-1, 784)
x_test_all  = x_test_all.reshape(-1, 784)

print(f'TensorFlow: {tf.__version__}')
print('✅ Setup complete.')

---
# Part A — Synthetic 2D Anomaly Detection

## Part A.1 — Generate the Dataset

We create a simple 2D dataset where:
- **Normal data** forms a Gaussian cluster centred at the origin
- **Anomalies** are scattered randomly in a wider region outside the normal cluster

The autoencoder will be trained **only on normal data**. At test time we present it with both normal and anomalous points and measure the reconstruction error for each.

> **Why start with 2D?** Because we can visualise every point, every reconstruction, and every error directly. The mechanism is exactly the same in high-dimensional settings — the 2D case just makes it impossible to miss.

In [ ]:
# Normal data — tight Gaussian cluster
N_NORMAL   = 1000
N_ANOMALY  = 50

normal_data   = np.random.normal(loc=0, scale=0.5, size=(N_NORMAL, 2)).astype('float32')
anomaly_data  = np.random.uniform(low=-3, high=3, size=(N_ANOMALY, 2)).astype('float32')
# Ensure anomalies are outside the normal region
far_mask = np.sqrt(anomaly_data[:,0]**2 + anomaly_data[:,1]**2) < 1.5
while far_mask.any():
    anomaly_data[far_mask] = np.random.uniform(-3, 3, (far_mask.sum(), 2))
    far_mask = np.sqrt(anomaly_data[:,0]**2 + anomaly_data[:,1]**2) < 1.5

# Combined test set with labels (0 = normal, 1 = anomaly)
all_data   = np.vstack([normal_data, anomaly_data])
all_labels = np.array([0]*N_NORMAL + [1]*N_ANOMALY)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(normal_data[:,0],  normal_data[:,1],
           c='steelblue', alpha=0.4, s=20, label=f'Normal  (n={N_NORMAL})')
ax.scatter(anomaly_data[:,0], anomaly_data[:,1],
           c='tomato', s=60, marker='x', linewidths=1.5, label=f'Anomaly (n={N_ANOMALY})')
ax.set_title('Synthetic Dataset — Normal Cluster + Anomalies', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

print(f'Normal data: {normal_data.shape}  — will be used for training')
print(f'Anomaly data: {anomaly_data.shape}  — never shown during training')
print(f'\nThe autoencoder will only ever see normal data during training.')
print('At test time we present both and see which produces high reconstruction error.')

## Part A.2 — Train on Normal Data Only

In [ ]:
# Tiny autoencoder for 2D data
synth_input  = keras.Input(shape=(2,))
encoded      = layers.Dense(8,  activation='relu')(synth_input)
bottleneck   = layers.Dense(1,  activation=None)(encoded)      # 1D bottleneck
decoded_h    = layers.Dense(8,  activation='relu')(bottleneck)
decoded_out  = layers.Dense(2,  activation=None)(decoded_h)   # linear output for 2D coords

synth_ae = Model(synth_input, decoded_out, name='synthetic_ae')
synth_ae.compile(optimizer='adam', loss='mse')  # MSE for continuous 2D data

print('Training on normal data ONLY...')
synth_ae.fit(
    normal_data, normal_data,    # input = target = normal points
    epochs=100,
    batch_size=64,
    verbose=0
)
print('✅ Training complete.')
print()
print(f'The model has learned a 1D bottleneck representation of a 2D Gaussian cluster.')
print(f'It has never seen the anomaly points during training.')

## Part A.3 — Reconstruction Error as Anomaly Score

We pass every point — normal and anomalous — through the trained autoencoder and measure how well it can reconstruct each one. Normal points that resemble the training data should reconstruct accurately. Anomalous points that fall outside the learned representation should not.

In [ ]:
# Reconstruct all points and compute per-point MSE
all_recon  = synth_ae.predict(all_data, verbose=0)
recon_mse  = np.mean((all_data - all_recon) ** 2, axis=1)

normal_errors  = recon_mse[all_labels == 0]
anomaly_errors = recon_mse[all_labels == 1]

print(f'Reconstruction MSE — Normal points:')
print(f'  Mean: {normal_errors.mean():.5f}  |  Max: {normal_errors.max():.5f}')
print(f'Reconstruction MSE — Anomaly points:')
print(f'  Mean: {anomaly_errors.mean():.5f}  |  Min: {anomaly_errors.min():.5f}')

# Scatter coloured by reconstruction error
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: scatter coloured by error magnitude
vmax = np.percentile(recon_mse, 98)
sc = ax1.scatter(all_data[:,0], all_data[:,1], c=recon_mse,
                 cmap='RdYlGn_r', s=30, vmin=0, vmax=vmax, alpha=0.7)
plt.colorbar(sc, ax=ax1, label='Reconstruction MSE (anomaly score)')
ax1.set_title('Scatter Coloured by Reconstruction Error\nGreen = low error, Red = high error',
              fontsize=11, fontweight='bold')
ax1.set_aspect('equal'); ax1.grid(alpha=0.2)

# Right: error distribution by class
ax2.hist(normal_errors,  bins=40, alpha=0.6, color='steelblue', label='Normal',  density=True)
ax2.hist(anomaly_errors, bins=40, alpha=0.6, color='tomato',    label='Anomaly', density=True)
ax2.set_xlabel('Reconstruction MSE (anomaly score)', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title('Error Distribution by Class\n(Ideal: well-separated distributions)',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)

plt.suptitle('Reconstruction Error as Anomaly Score', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## Part A.4 — Set a Threshold and Evaluate

A reconstruction error score is not useful on its own — we need a **threshold** above which a point is flagged as anomalous. The threshold is a design decision that trades off precision (how many flagged points are true anomalies) against recall (how many true anomalies are flagged).

> Use the slider to explore this trade-off directly.

In [ ]:
# Evaluate across a range of thresholds
thresholds  = np.linspace(recon_mse.min(), recon_mse.max(), 200)
precisions, recalls, f1s = [], [], []

for thresh in thresholds:
    predicted_anomaly = (recon_mse >= thresh).astype(int)
    tp = ((predicted_anomaly == 1) & (all_labels == 1)).sum()
    fp = ((predicted_anomaly == 1) & (all_labels == 0)).sum()
    fn = ((predicted_anomaly == 0) & (all_labels == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    precisions.append(prec); recalls.append(rec); f1s.append(f1)

best_thresh = thresholds[np.argmax(f1s)]
best_f1     = max(f1s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall vs Threshold
ax1.plot(thresholds, precisions, label='Precision', color='steelblue', linewidth=2)
ax1.plot(thresholds, recalls,    label='Recall',    color='tomato',    linewidth=2)
ax1.plot(thresholds, f1s,        label='F1',        color='seagreen',  linewidth=2)
ax1.axvline(best_thresh, color='black', linestyle='--', linewidth=1.5,
            label=f'Best F1 threshold = {best_thresh:.4f}')
ax1.set_xlabel('Threshold'); ax1.set_ylabel('Score')
ax1.set_title('Precision, Recall, F1 vs Threshold', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# Show detections at best threshold
flagged = recon_mse >= best_thresh
ax2.scatter(all_data[~flagged & (all_labels==0), 0],
            all_data[~flagged & (all_labels==0), 1],
            c='steelblue', alpha=0.4, s=20, label='True Normal (correctly not flagged)')
ax2.scatter(all_data[flagged & (all_labels==1), 0],
            all_data[flagged & (all_labels==1), 1],
            c='tomato', s=80, marker='x', linewidths=2, label='True Anomaly (correctly flagged)')
ax2.scatter(all_data[flagged & (all_labels==0), 0],
            all_data[flagged & (all_labels==0), 1],
            c='orange', s=50, marker='o', alpha=0.7, label='False Positive (normal flagged)')
ax2.scatter(all_data[~flagged & (all_labels==1), 0],
            all_data[~flagged & (all_labels==1), 1],
            c='purple', s=80, marker='s', alpha=0.7, label='False Negative (anomaly missed)')
ax2.set_title(f'Detections at Threshold = {best_thresh:.4f}  (Best F1 = {best_f1:.3f})',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(alpha=0.2); ax2.set_aspect('equal')

plt.tight_layout(); plt.show()

print(f'At best F1 threshold ({best_thresh:.4f}):')
tp_best = ((recon_mse >= best_thresh) & (all_labels == 1)).sum()
fp_best = ((recon_mse >= best_thresh) & (all_labels == 0)).sum()
fn_best = ((recon_mse < best_thresh)  & (all_labels == 1)).sum()
print(f'  True positives (anomalies caught): {tp_best} / {N_ANOMALY}')
print(f'  False positives (normal flagged):  {fp_best}')
print(f'  False negatives (anomalies missed):{fn_best}')

---
# Part B — MNIST One-Class Anomaly Detection

## Part B.1 — Select the Normal Class

We treat images of one digit class as *normal* and all other digit classes as *anomalies*. The autoencoder is trained only on the normal class. At test time we evaluate reconstruction error across all digit classes and measure how well the model separates normal from anomalous.

**Change `NORMAL_DIGIT` below to experiment with different normal classes.**

In [ ]:
# ── Choose which digit is 'normal' ──
NORMAL_DIGIT = 0   # change this to 1, 2, 3... to experiment

# Training data: only images of NORMAL_DIGIT
train_mask   = y_train == NORMAL_DIGIT
x_train_normal = x_train_all[train_mask]

# Test data: all digits — binary labels (0=normal, 1=anomaly)
y_test_binary = (y_test != NORMAL_DIGIT).astype(int)  # 0 if normal, 1 if anomaly

print(f'Normal class: digit {NORMAL_DIGIT}')
print(f'Training images (digit {NORMAL_DIGIT} only): {x_train_normal.shape[0]:,}')
print(f'Test images (all digits):  {x_test_all.shape[0]:,}')
print(f'  Normal ({NORMAL_DIGIT}s) in test set:  {(y_test==NORMAL_DIGIT).sum():,}')
print(f'  Anomalies (other digits): {(y_test!=NORMAL_DIGIT).sum():,}')
print()

# Preview the normal class
fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    ax.imshow(x_train_normal[i].reshape(28,28), cmap='gray')
    ax.axis('off')
plt.suptitle(f'Training data — digit {NORMAL_DIGIT} only (10 examples shown)',
             fontsize=10, y=1.1)
plt.tight_layout(); plt.show()

## Part B.2 — Train on the Normal Class Only

In [ ]:
LATENT_DIM = 32

enc_in  = keras.Input(shape=(784,))
enc_h   = layers.Dense(128, activation='relu')(enc_in)
enc_out = layers.Dense(LATENT_DIM, activation=None)(enc_h)
encoder = Model(enc_in, enc_out)

dec_in  = keras.Input(shape=(LATENT_DIM,))
dec_h   = layers.Dense(128, activation='relu')(dec_in)
dec_out = layers.Dense(784, activation='sigmoid')(dec_h)
decoder = Model(dec_in, dec_out)

ae_in   = keras.Input(shape=(784,))
ae_out  = decoder(encoder(ae_in))
oneclass_ae = Model(ae_in, ae_out, name=f'oneclass_ae_digit{NORMAL_DIGIT}')
oneclass_ae.compile(optimizer='adam', loss='binary_crossentropy')

print(f'Training one-class autoencoder on digit {NORMAL_DIGIT} only...')
hist = oneclass_ae.fit(
    x_train_normal, x_train_normal,
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_split=0.1,
    verbose=1
)
print('\n✅ Training complete.')
print(f'The model has learned to reconstruct only digit {NORMAL_DIGIT}.')
print('It has never seen any other digit class during training.')

## Part B.3 — Reconstruction Error Across All Digit Classes

We pass all 10,000 test images through the model and compute the reconstruction error per image. If the approach works, digit `NORMAL_DIGIT` should have low error (normal = reconstructed well) while all other digits have higher error (anomalous = reconstructed poorly).

In [ ]:
# Reconstruct all test images
recon_all   = oneclass_ae.predict(x_test_all, verbose=0)
errors_all  = np.mean((x_test_all - recon_all) ** 2, axis=1)

# Per-class error statistics
print(f'Reconstruction MSE per digit class (trained on digit {NORMAL_DIGIT} only):')
print(f'{"Digit":>8}  {"Mean MSE":>12}  {"Median MSE":>12}  {"Label":>10}')
print('-' * 50)

for digit in range(10):
    mask       = y_test == digit
    mean_err   = errors_all[mask].mean()
    median_err = np.median(errors_all[mask])
    label      = '← NORMAL' if digit == NORMAL_DIGIT else ''
    print(f'{digit:>8}  {mean_err:>12.5f}  {median_err:>12.5f}  {label}')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot
data_by_class = [errors_all[y_test == d] for d in range(10)]
bp = axes[0].boxplot(data_by_class, patch_artist=True, medianprops={'color':'black','linewidth':2})
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('steelblue' if i == NORMAL_DIGIT else 'salmon')
    patch.set_alpha(0.7)
axes[0].set_xlabel('Digit Class', fontsize=11)
axes[0].set_ylabel('Reconstruction MSE', fontsize=11)
axes[0].set_title(f'Error Distribution per Class\n(Blue = normal class: digit {NORMAL_DIGIT})',
                  fontsize=11, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')
axes[0].set_xticklabels([str(d) for d in range(10)])

# Histogram: normal vs all anomalies
normal_err  = errors_all[y_test == NORMAL_DIGIT]
anomaly_err = errors_all[y_test != NORMAL_DIGIT]
axes[1].hist(normal_err,  bins=50, alpha=0.6, color='steelblue', label=f'Digit {NORMAL_DIGIT} (normal)',  density=True)
axes[1].hist(anomaly_err, bins=50, alpha=0.4, color='salmon',    label='All other digits (anomaly)', density=True)
axes[1].set_xlabel('Reconstruction MSE', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Error Distribution: Normal vs Anomaly', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle(f'One-Class Anomaly Detection — Digit {NORMAL_DIGIT} as Normal Class',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# Visualise best and worst reconstructions per class
print(f'\nSample reconstructions — digit {NORMAL_DIGIT} (normal) vs digit {(NORMAL_DIGIT+1)%10} (anomaly):')
fig, axes = plt.subplots(4, 8, figsize=(14, 7.5))
anomaly_digit = (NORMAL_DIGIT + 1) % 10
normal_idx    = np.where(y_test == NORMAL_DIGIT)[0][:8]
anomaly_idx   = np.where(y_test == anomaly_digit)[0][:8]

for i in range(8):
    axes[0,i].imshow(x_test_all[normal_idx[i]].reshape(28,28), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(recon_all[normal_idx[i]].reshape(28,28), cmap='gray'); axes[1,i].axis('off')
    axes[2,i].imshow(x_test_all[anomaly_idx[i]].reshape(28,28), cmap='gray'); axes[2,i].axis('off')
    axes[3,i].imshow(recon_all[anomaly_idx[i]].reshape(28,28), cmap='gray'); axes[3,i].axis('off')

axes[0,0].set_ylabel(f'Digit {NORMAL_DIGIT}\n(original)', fontsize=8, rotation=0, labelpad=60, va='center')
axes[1,0].set_ylabel(f'Digit {NORMAL_DIGIT}\n(recon)', fontsize=8, rotation=0, labelpad=60, va='center')
axes[2,0].set_ylabel(f'Digit {anomaly_digit}\n(original)', fontsize=8, rotation=0, labelpad=60, va='center')
axes[3,0].set_ylabel(f'Digit {anomaly_digit}\n(recon)', fontsize=8, rotation=0, labelpad=60, va='center')

plt.suptitle(f'Normal class (digit {NORMAL_DIGIT}) reconstructs cleanly — '
             f'anomaly class (digit {anomaly_digit}) does not',
             fontsize=10, y=1.01)
plt.tight_layout(); plt.show()

## Part B.4 — ROC Curve and AUC

The ROC curve shows how the true positive rate (anomalies correctly flagged) trades off against the false positive rate (normal images incorrectly flagged) across all possible thresholds. AUC summarises this as a single number: 0.5 = random, 1.0 = perfect.

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test_binary, errors_all)
roc_auc = auc(fpr, tpr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
ax1.plot(fpr, tpr, color='steelblue', linewidth=2.5,
         label=f'ROC curve  (AUC = {roc_auc:.3f})')
ax1.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random classifier (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title(f'ROC Curve — Digit {NORMAL_DIGIT} vs All Others', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10); ax1.grid(alpha=0.3)

# AUC per anomaly class
auc_per_digit = {}
for digit in range(10):
    if digit == NORMAL_DIGIT:
        continue
    mask     = (y_test == NORMAL_DIGIT) | (y_test == digit)
    labels_d = (y_test[mask] != NORMAL_DIGIT).astype(int)
    errors_d = errors_all[mask]
    f, t, _  = roc_curve(labels_d, errors_d)
    auc_per_digit[digit] = auc(f, t)

digits_sorted = sorted(auc_per_digit.keys())
auc_vals      = [auc_per_digit[d] for d in digits_sorted]
bar_colors    = ['tomato' if a < 0.7 else ('orange' if a < 0.85 else 'steelblue') for a in auc_vals]

ax2.bar([str(d) for d in digits_sorted], auc_vals, color=bar_colors, edgecolor='white')
ax2.axhline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Random baseline')
ax2.set_xlabel('Anomaly digit class', fontsize=11)
ax2.set_ylabel('AUC', fontsize=11)
ax2.set_title(f'AUC per Anomaly Class\n(vs normal class: digit {NORMAL_DIGIT})',
              fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=9); ax2.grid(alpha=0.3, axis='y')
for i, (d, v) in enumerate(zip(digits_sorted, auc_vals)):
    ax2.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle(f'Anomaly Detection Performance — Normal class: digit {NORMAL_DIGIT}',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print(f'Overall AUC (digit {NORMAL_DIGIT} vs all others): {roc_auc:.3f}')
print()
print('AUC interpretation: 0.5 = random, 0.7 = fair, 0.8 = good, 0.9+ = excellent')
print()
print('📊 Which digit classes are hardest to detect as anomalies?')
print(f'   Think about which digits look most similar to digit {NORMAL_DIGIT}.')
print('   Does the AUC pattern match your visual intuition?')

## Part B.5 — Limitations Discussion

The reading pages for this lesson identified three key limitations of reconstruction-based anomaly detection. The questions below ask you to engage with each one directly, using what you observed in this notebook.

---

### ❓ Discussion Question 1 — The Capacity Trade-off

**The reading warned that if the autoencoder is too expressive (large bottleneck, many layers, long training), it may learn to reconstruct anomalies accurately — losing its discriminative power.**

**Change `LATENT_DIM` from 32 to 256 and retrain (Part B.2 onwards). Does the AUC increase or decrease? Does the reconstruction error gap between normal and anomalous classes narrow? What does this tell you about the trade-off between reconstruction quality and anomaly detection performance?**

> **Your answer:** *(Replace this text)*

---

### ❓ Discussion Question 2 — Visually Similar Anomalies

**Look at the per-anomaly-class AUC bar chart. Which anomaly digit class has the lowest AUC — meaning the autoencoder finds it hardest to distinguish from the normal class?**

**Display a sample of images from that digit class alongside the normal class. Can you see visually why they are hard to separate? What does this tell you about the kind of anomalies reconstruction-based detection will reliably catch versus the ones it will miss?**

In [ ]:
# Find the hardest anomaly class
hardest_digit = min(auc_per_digit, key=auc_per_digit.get)
print(f'Hardest anomaly class to detect: digit {hardest_digit}  (AUC = {auc_per_digit[hardest_digit]:.3f})')
print(f'Normal class: digit {NORMAL_DIGIT}')

fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
normal_idxs  = np.where(y_test == NORMAL_DIGIT)[0][:10]
hardest_idxs = np.where(y_test == hardest_digit)[0][:10]

for i in range(10):
    axes[0,i].imshow(x_test_all[normal_idxs[i]].reshape(28,28), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(x_test_all[hardest_idxs[i]].reshape(28,28), cmap='gray'); axes[1,i].axis('off')

axes[0,0].set_ylabel(f'Digit {NORMAL_DIGIT}\n(normal)', fontsize=8, rotation=0, labelpad=55, va='center')
axes[1,0].set_ylabel(f'Digit {hardest_digit}\n(hardest anomaly)', fontsize=8, rotation=0, labelpad=55, va='center')
plt.suptitle(f'Why digit {hardest_digit} is hard to detect as anomaly when normal = digit {NORMAL_DIGIT}',
             fontsize=10, y=1.05)
plt.tight_layout(); plt.show()

> **Your answer to Discussion Question 2:** *(Replace this text)*

---

### ❓ Discussion Question 3 — Real-World Deployment

**The reading said: *"An anomaly detection system deployed in the real world should never be the sole decision-maker. Reconstruction error is a ranking signal, not a binary verdict."***

**Based on what you have seen in this notebook, why is this advice important? What would the consequences be of acting automatically on the reconstruction error threshold without human review in a high-stakes setting — such as medical diagnosis or fraud detection?**

**In your chosen high-stakes setting, would you prioritise minimising false positives or false negatives? How would this change where you set the threshold?**

> **Your answer:** *(Replace this text)*

---
## ✅ Notebook 2 Complete

| ✅ | Achievement |
|----|-------------|
| ✅ | Demonstrated reconstruction-based anomaly detection on synthetic 2D data |
| ✅ | Visualised reconstruction error as a colour-coded anomaly score on a scatter plot |
| ✅ | Set and evaluated an anomaly threshold — measuring TP, FP, FN and the precision-recall trade-off |
| ✅ | Trained a one-class MNIST autoencoder on a single digit class |
| ✅ | Computed reconstruction error across all 10 digit classes and identified which are hardest to detect |
| ✅ | Evaluated detection performance using ROC curves and per-class AUC |
| ✅ | Engaged with three honest limitations of the approach through guided discussion |

---

### ➡️ You have completed all notebook activities for Module 3

Return to Moodle to complete the **end-of-module graded quiz** — the assessed activity for this module, covering key concepts from all three lessons.